In [2]:
!pip install torch

  Using cached torch-2.10.0-2-cp313-none-macosx_11_0_arm64.whl.metadata (31 kB)
  Using cached filelock-3.25.2-py3-none-any.whl.metadata (2.0 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached setuptools-82.0.1-py3-none-any.whl.metadata (6.5 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached fsspec-2026.2.0-py3-none-any.whl.metadata (10 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
  Using cached markupsafe-3.0.3-cp313-cp313-macosx_11_0_arm64.whl.metadata (2.7 kB)
Using cached torch-2.10.0-2-cp313-none-macosx_11_0_arm64.whl (79.5 MB)
Using cached fsspec-2026.2.0-py3-none-any.whl (202 kB)
Using cached networkx-3.6.1-py3-none-any.whl (2.1 MB)
Using cached sympy-1.14.0-py3-none-any.whl (6.3 MB)
Using cached mpmath-1.3.0-py3-none-any.whl (536 kB)
Using cached typing_exten

## Differentiation Matrix

In [3]:
import torch

def chebyshev_cgl(N, device="cpu"):
    j = torch.arange(0, N+1, device=device)
    x = torch.cos(torch.pi * j / N)
    return x


def chebyshev_diff_matrix(N, device="cpu"):
    x = chebyshev_cgl(N, device)
    
    c = torch.ones(N+1, device=device)
    c[0] = 2
    c[-1] = 2
    
    X = x.unsqueeze(0)
    dX = X.T - X  # x_i - x_j
    
    C = c.unsqueeze(0)
    C_ratio = C.T / C
    
    D = C_ratio * ((-1)**(torch.arange(N+1).unsqueeze(0) + torch.arange(N+1).unsqueeze(1))) / (dX + torch.eye(N+1, device=device))
    
    # Fix diagonal separately
    D = D - torch.diag(torch.sum(D, dim=1))
    
    return x, D

/Users/jishnusatwikkancherlapalli/Documents/Developer/PINN(Physics Informed Neural Networks)/.venv/lib/python3.13/site-packages/torch/_subclasses/functional_tensor.py:283: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


### verification of the differentiation matrix

$$u(x) = \sin(\pi x)$$
$$u'(x) = \pi \cos(\pi x)$$

In [7]:
N = 20
x, D = chebyshev_diff_matrix(N)

u = torch.sin(torch.pi * x)
ux_true = torch.pi * torch.cos(torch.pi * x)

ux_pred = D @ u

error = torch.norm(ux_pred - ux_true)
print("Derivative error:", error.item())

Derivative error: 4.833216280530905e-06
